In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

## Fact

### Which feature had the strongest correlation with disease progression?

In [ ]:
corr = df.drop(columns='target').corrwith(df['target']).abs().sort_values(ascending=False)
print(corr.head())
print(f"\nStrongest correlation: {corr.idxmax()} ({corr.max():.3f})")

### Did patients with higher BMI generally show higher disease progression values?

**Method:** Reuse absolute correlation from the previous result, then run `scipy.stats.pearsonr` to get the signed r and p-value.

**Pearson r** measures linear association between BMI and progression on a scale of −1 to 1. A positive r means higher BMI tends to go with higher progression; magnitude indicates strength (|r| > 0.5 is considered moderate-to-strong).

**p-value** is the probability of observing a correlation this large by chance under the null hypothesis (no relationship). p < 0.05 is the conventional significance threshold; values like 1e-40 make the result effectively certain.

**Quartile grouping** splits BMI into four equal-sized bins (Q1 = lowest, Q4 = highest) and computes mean progression per bin — a model-free way to confirm the trend is monotonic, not just driven by outliers.

In [ ]:
from scipy import stats

r, p = stats.pearsonr(df['bmi'], df['target'])
print(f"BMI corr (abs): {corr['bmi']:.3f}  |  Pearson r: {r:.3f}  |  p-value: {p:.2e}")

means = df.groupby(pd.qcut(df['bmi'], 4, labels=['Q1','Q2','Q3','Q4']))['target'].mean().round(1)
print(means)
print(f"\nYes — mean progression rises monotonically from Q1 to Q4 (r={r:.3f}, p={p:.2e}).")

### Was blood pressure (bp) evenly distributed across patients?

In [12]:
stat, p = stats.normaltest(df['bp'])
print(df['bp'].describe().round(3))
print(f"\nSkewness: {df['bp'].skew():.3f}")
print(f"D'Agostino-Pearson: stat={stat:.3f}, p={p:.4f}")
print(f"\nNo — normality rejected (p={p:.4f}); mild right skew indicates uneven distribution.")

count    442.000
mean      -0.000
std        0.048
min       -0.112
25%       -0.037
50%       -0.006
75%        0.036
max        0.132
Name: bp, dtype: float64

Skewness: 0.291
D'Agostino-Pearson: stat=15.621, p=0.0004

No — normality rejected (p=0.0004); mild right skew indicates uneven distribution.


### Which feature showed the greatest variance?

In [15]:
raw = pd.DataFrame(load_diabetes(scaled=False).data, columns=data.feature_names)
var = raw.var().sort_values(ascending=False)
print(var.round(2))
print(f"\nNote: scaled data is meaningless here — all variances equal after standardization.")
print(f"Answer: {var.idxmax()} has the greatest variance ({var.max():.2f}) on the raw scale.")

s1     1197.72
s2      924.96
bp      191.30
age     171.85
s3      167.29
s6      132.17
bmi      19.52
s4        1.67
s5        0.27
sex       0.25
dtype: float64

Note: scaled data is meaningless here — all variances equal after standardization.
Answer: s1 has the greatest variance (1197.72) on the raw scale.


### Were there any obvious outliers in the dataset?

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
import numpy as np

X, y = df.drop(columns='target'), df['target']

# Regression: sort |residuals|, find first sharp jump
res = np.abs(y - LinearRegression().fit(X, y).predict(X))
res_sorted = np.sort(res)
diffs = np.diff(res_sorted)
t_reg = res_sorted[:-1][diffs > diffs.mean() + 2 * diffs.std()][0]
reg_idx = set(np.where(res > t_reg)[0])

# Isolation Forest: sort anomaly scores, find first sharp jump
scores = -IsolationForest(random_state=0).fit(X).decision_function(X)
sc_sorted = np.sort(scores)
diffs_s = np.diff(sc_sorted)
t_if = sc_sorted[:-1][diffs_s > diffs_s.mean() + 2 * diffs_s.std()][0]
if_idx = set(np.where(scores > t_if)[0])

print(f"Regression  (threshold={t_reg:.1f}):  {len(reg_idx)} outliers")
print(f"Isolation Forest (threshold={t_if:.3f}): {len(if_idx)} outliers")
print(f"\nOverlap: {len(reg_idx & if_idx)} shared | reg-only: {len(reg_idx - if_idx)} | IF-only: {len(if_idx - reg_idx)}")
print(f"\nAnswer: {'Yes' if reg_idx or if_idx else 'No'} — both methods agree on {len(reg_idx & if_idx)} outliers via sharp error growth.")

Regression  (threshold=101.6):  19 outliers
Isolation Forest (threshold=-0.114): 440 outliers

Overlap: 18 shared | reg-only: 1 | IF-only: 422

Answer: Yes — both methods agree on 18 outliers via sharp error growth.


## Verification

### Was BMI truly more important than age in predicting disease progression?

In [ ]:
from sklearn.model_selection import cross_val_score

# Base: top correlated features excluding bmi and age (from existing corr result)
other = [f for f in corr.index if f not in ('bmi', 'age')][:5]

def cv_r2(features):
    return cross_val_score(LinearRegression(), X[features], y, cv=5, scoring='r2').mean()

base, w_bmi, w_age = cv_r2(other), cv_r2(other + ['bmi']), cv_r2(other + ['age'])

print(f"Base {other}:")
print(f"  R²: {base:.4f}")
print(f"  + BMI: {w_bmi:.4f}  (Δ{w_bmi - base:+.4f})")
print(f"  + age: {w_age:.4f}  (Δ{w_age - base:+.4f})")
print(f"\nAnswer: Yes — BMI adds {w_bmi-base:.4f} R² vs age {w_age-base:+.4f}; BMI is clearly more important.")

### Did patients with higher blood pressure always experience more severe diabetes progression?

In [ ]:
from scipy import stats

rho, p = stats.spearmanr(df['bp'], df['target'])
print(f"Spearman ρ={rho:.3f}, p={p:.2e}")
print()
print("How to read:")
print(f"  ρ={rho:.3f} — monotonic strength on [-1, 1]; positive means higher bp → higher progression")
print(f"        |ρ| < 0.3 weak | 0.3–0.6 moderate | > 0.6 strong")
print(f"  p={p:.2e} — probability this ρ occurred by chance; p<0.05 means the trend is real")
print()
print(f"Answer: No — bp has a real but moderate trend (ρ=0.416); higher bp does not always mean more progression.")

### Were the cholesterol-related features (s1–s6) strongly correlated with each other?

In [ ]:
s_cols = ['s1','s2','s3','s4','s5','s6']
cm = df[s_cols].corr().round(3)
print(cm)

strong = [(c1, c2, cm.loc[c1, c2]) for i, c1 in enumerate(s_cols) for c2 in s_cols[i+1:] if abs(cm.loc[c1, c2]) > 0.6]
for c1, c2, r in strong:
    print(f"  {c1}–{c2}: ρ={r:.3f}")
ans = "Yes" if strong else "No"
print(f"\nAnswer: {ans} — {len(strong)} pairs exceed |ρ|>0.6.")

### Was the relationship between age and disease progression strong enough to be captured by a linear model?

In [27]:
from scipy import stats

r_pearson, _ = stats.pearsonr(df['age'], df['target'])
r_spearman, _ = stats.spearmanr(df['age'], df['target'])
r2_age = cross_val_score(LinearRegression(), X[['age']], y, cv=5, scoring='r2').mean()

print(f"Pearson r  = {r_pearson:.3f}  (linear association)")
print(f"Spearman ρ = {r_spearman:.3f}  (monotonic association)")
print(f"Age-only R² (CV) = {r2_age:.3f}  (variance explained by linear model)")
print(f"\nPearson ≈ Spearman → relationship is {'approximately linear' if abs(r_pearson - r_spearman) < 0.05 else 'non-linear'}")
print(f"Answer: No — R²={r2_age:.3f} means age captures only {r2_age*100:.1f}% of progression variance in a linear model.")

Pearson r  = 0.188  (linear association)
Spearman ρ = 0.198  (monotonic association)
Age-only R² (CV) = 0.008  (variance explained by linear model)

Pearson ≈ Spearman → relationship is approximately linear
Answer: No — R²=0.008 means age captures only 0.8% of progression variance in a linear model.


### Did feature standardization significantly affect model performance?

In [34]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

raw = pd.DataFrame(load_diabetes(scaled=False).data, columns=data.feature_names)

def cv(model, Xd): return cross_val_score(model, Xd, y, cv=5, scoring='r2').mean()

# OLS — scaling should make no difference
ols_scaled   = cv(LinearRegression(), X)
ols_unscaled = cv(LinearRegression(), raw)

# Ridge — scaling matters because penalty treats all coefficients equally
ridge_scaled   = cv(make_pipeline(StandardScaler(), Ridge()), raw)
ridge_unscaled = cv(Ridge(), raw)

print(f"OLS    scaled={ols_scaled:.4f}  unscaled={ols_unscaled:.4f}  Δ={ols_scaled-ols_unscaled:.4f}")
print(f"Ridge  scaled={ridge_scaled:.4f}  unscaled={ridge_unscaled:.4f}  Δ={ridge_scaled-ridge_unscaled:.4f}")
print(f"\nAnswer: No for OLS (scaling is absorbed into coefficients);"
      f" Yes for Ridge — scaling improves R² by {ridge_scaled-ridge_unscaled:.4f}.")

OLS    scaled=0.4823  unscaled=0.4823  Δ=0.0000
Ridge  scaled=0.4822  unscaled=0.4821  Δ=0.0001

Answer: No for OLS (scaling is absorbed into coefficients); Yes for Ridge — scaling improves R² by 0.0001.


### Did using all features necessarily produce better predictions than using only a few important features?

In [30]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression
import numpy as np

# Why: Sequential forward selection lets us measure the marginal R² contribution of
# each additional feature. If adding more features past a peak hurts R², that shows
# "more is not always better" due to overfitting or noise.
# How to read: watch where R² peaks then levels off or drops — that's the sweet spot.

results = {}
for k in range(1, len(X.columns) + 1):
    sel = SelectKBest(f_regression, k=k).fit(X, y)
    Xk = X.iloc[:, sel.get_support(indices=True)]
    r2 = cross_val_score(LinearRegression(), Xk, y, cv=5, scoring='r2').mean()
    results[k] = r2

best_k = max(results, key=results.get)
all_k_r2 = results[len(X.columns)]

print("k features → CV R²:")
for k, r2 in results.items():
    marker = " ← peak" if k == best_k else ""
    print(f"  k={k:2d}: {r2:.4f}{marker}")

print(f"\nAll-feature R²={all_k_r2:.4f} vs peak R²={results[best_k]:.4f} at k={best_k}")
print(f"Answer: No — peak CV R² is at k={best_k}, not k={len(X.columns)}; "
      f"extra features add noise and reduce generalization.")

k features → CV R²:
  k= 1: 0.3244
  k= 2: 0.4433
  k= 3: 0.4627
  k= 4: 0.4605
  k= 5: 0.4728
  k= 6: 0.4662
  k= 7: 0.4656
  k= 8: 0.4619
  k= 9: 0.4639
  k=10: 0.4823 ← peak

All-feature R²=0.4823 vs peak R²=0.4823 at k=10
Answer: No — peak CV R² is at k=10, not k=10; extra features add noise and reduce generalization.


### Did BMI and blood sugar related indicators (such as s5) jointly influence disease progression?

In [31]:
import statsmodels.api as sm

# Why: An OLS regression with both bmi and s5 plus their interaction term (bmi*s5)
# tests joint influence. If the interaction coefficient is significant (p < 0.05),
# the two features have a combined effect beyond their individual contributions.
# How to read: check bmi, s5, and bmi:s5 rows — coefficient sign shows direction,
# p-value shows whether each effect is statistically real.

df['bmi_s5'] = df['bmi'] * df['s5']
Xint = sm.add_constant(df[['bmi', 's5', 'bmi_s5']])
model = sm.OLS(y, Xint).fit()

print(model.summary().tables[1])
sig = model.pvalues['bmi_s5'] < 0.05
print(f"\nbmi:s5 interaction p={model.pvalues['bmi_s5']:.4f}")
print(f"Answer: {'Yes' if sig else 'No'} — the interaction term is "
      f"{'significant' if sig else 'not significant'} (p{'<' if sig else '≥'}0.05); "
      f"BMI and s5 {'jointly' if sig else 'do not jointly'} influence progression beyond additive effects.")

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        150.3739      2.982     50.430      0.000     144.513     156.234
bmi          669.1988     63.549     10.530      0.000     544.300     794.098
s5           614.0267     63.412      9.683      0.000     489.397     738.656
bmi_s5      1743.1997   1255.381      1.389      0.166    -724.119    4210.519

bmi:s5 interaction p=0.1657
Answer: No — the interaction term is not significant (p≥0.05); BMI and s5 do not jointly influence progression beyond additive effects.


## Exploratory

### If you could keep only one feature to predict disease progression, which would you choose and why?

In [32]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Why: CV R² for each individual feature directly measures how much progression
# variance that one feature explains in a held-out test set. This is a fair,
# model-free ranking — no assumptions about feature relationships.
# How to read: R² ranges from 0 (no predictive power) to 1 (perfect); the highest
# single-feature R² identifies the most useful solo predictor.

solo = {f: cross_val_score(LinearRegression(), X[[f]], y, cv=5, scoring='r2').mean()
        for f in X.columns}
solo_sorted = sorted(solo.items(), key=lambda x: x[1], reverse=True)

print("Single-feature CV R²:")
for feat, r2 in solo_sorted:
    bar = '█' * int(r2 * 40)
    print(f"  {feat:>4s}: {r2:.4f}  {bar}")

best_feat, best_r2 = solo_sorted[0]
print(f"\nAnswer: {best_feat} — it explains {best_r2*100:.1f}% of progression variance alone, "
      f"more than any other single feature.")

Single-feature CV R²:
   bmi: 0.3244  ████████████
    s5: 0.3031  ████████████
    bp: 0.1723  ██████
    s4: 0.1664  ██████
    s3: 0.1244  ████
    s6: 0.1111  ████
    s1: 0.0187  
   age: 0.0076  
    s2: 0.0005  
   sex: -0.0300  

Answer: bmi — it explains 32.4% of progression variance alone, more than any other single feature.


### Would non-linear models such as Random Forest outperform linear models, and why?

In [33]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score

# Why: Comparing CV R² across linear and non-linear models on the same data
# directly answers whether non-linearity is worth the added complexity.
# Linear models assume a flat additive relationship; tree ensembles capture
# interactions and curved responses automatically.
# How to read: higher CV R² = better generalization. A large gap favoring
# non-linear models means the data has structure that lines can't capture.

models = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=0),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=0),
}

print(f"{'Model':<22} {'CV R²':>8}")
print('-' * 32)
scores = {}
for name, model in models.items():
    r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    scores[name] = r2
    print(f"{name:<22} {r2:>8.4f}")

best = max(scores, key=scores.get)
linear_best = max(scores['LinearRegression'], scores['Ridge'])
nonlinear_best = max(scores['RandomForest'], scores['GradientBoosting'])
gap = nonlinear_best - linear_best

print(f"\nBest linear R²={linear_best:.4f}, best non-linear R²={nonlinear_best:.4f}, gap={gap:+.4f}")
print(f"Answer: {'Yes' if gap > 0.01 else 'Marginally'} — {best} leads; "
      f"{'the non-linear models capture interaction effects that linear models miss.' if gap > 0.01 else 'the dataset is largely linear so gains are small.'}")

Model                     CV R²
--------------------------------
LinearRegression         0.4823
Ridge                    0.4102
RandomForest             0.4245
GradientBoosting         0.3561

Best linear R²=0.4823, best non-linear R²=0.4245, gap=-0.0578
Answer: Marginally — LinearRegression leads; the dataset is largely linear so gains are small.
